In [2]:
import pandas as pd
import numpy as np

In [4]:
sail_cluster = pd.read_excel('Sail_Cluster.xlsx')
sail_cluster.head()

,adjusted_date,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,Cluster
0,2024-01-01,9.71,11.4,2,17.0,0,3,0,1
1,2024-01-01,10.00,11.4,2,1.0,0,3,0,1
2,2024-01-02,8.57,11.4,2,0.7,0,1,0,1
3,2024-01-02,9.50,11.4,2,22.0,0,2,0,1
4,2024-01-03,10.42,11.4,2,2.4,0,3,0,1


In [6]:
sail_cluster['Cluster'].value_counts()

Cluster
1    560
Name: count, dtype: int64

In [8]:
port_cluster = pd.read_excel('Port_Cluster.xlsx')
port_cluster.head()

,adjusted_date,mean_draft,ae_t_steaming,aux_running,blr_running,activity_time,Cluster
0,2024-01-01,11.40,0,3,0,5.6,1
1,2024-01-03,11.40,0,1,0,17.3,1
2,2024-01-04,10.55,0,3,0,5.5,1
3,2024-01-07,8.40,0,3,0,18.8,1
4,2024-01-08,8.40,0,1,0,10.5,1


In [10]:
port_cluster.shape

(193, 7)

In [12]:
port_cluster['Cluster'].value_counts()

Cluster
1    193
Name: count, dtype: int64

In [14]:
sail_cluster[sail_cluster['Cluster'] == 1]['me_actual_steaming_time'].sum()/24, sail_cluster[sail_cluster['Cluster'] == 2]['me_actual_steaming_time'].sum()/24, sail_cluster[sail_cluster['Cluster'] == 3]['me_actual_steaming_time'].sum()/24

(188.9375, 0.0, 0.0)

In [16]:
import warnings
warnings.filterwarnings('ignore')

In [18]:
def find_sail_subset_data(df, expected_days): 
    target_sum = expected_days * 24
    while True:
        # Shuffle the DataFrame
        df_shuffled = df.sample(frac=1).reset_index(drop=True)



        df_sample = pd.DataFrame(columns=df.columns)
        current_sum = 0



        # Iterate over the shuffled DataFrame
        for index, row in df_shuffled.iterrows():
            if current_sum + row['me_actual_steaming_time'] <= target_sum:
                df_sample = df_sample.append(row)
                current_sum =   current_sum + row['me_actual_steaming_time'] 
#                             print(current_sum)

            if round(current_sum) == target_sum:
                 
                return df_sample

In [20]:
def find_port_subset_data(df, expected_days):
    target_sum = expected_days * 24
    while True:
        # Shuffle the DataFrame
        df_shuffled = df.sample(frac=1).reset_index(drop=True)



        df_sample = pd.DataFrame(columns=df.columns)
        current_sum = 0



        # Iterate over the shuffled DataFrame
        for index, row in df_shuffled.iterrows():
            if current_sum + row['activity_time'] <= target_sum:
                df_sample = df_sample.append(row)
                current_sum =   current_sum + row['activity_time'] 
                # print(current_sum)

            if round(current_sum) == target_sum:
                 
                return df_sample

In [22]:
def get_random_sample_with_time_sum(cluster_data, activity,  no_of_profiles,   expected_days  ):
    global df_sample
    target_sum = expected_days*24
    
    global   finaldict_sail, finaldict_port, final_sail_df, final_port_df
    
    finaldict_sail = { }
    finaldict_port = { }
    
    
    if activity == 'SAIL':
        final_sail_df = pd.DataFrame()
        for idx in range(1, no_of_profiles+1):
            print("idx-s", idx)
            df = cluster_data[cluster_data['Cluster'] == idx].copy() 
            print("Data Availibility Days", round(df['me_actual_steaming_time'].sum()/24))
            subset_df = find_sail_subset_data(df, expected_days)
            finaldict_sail['Subset_Cluster_Sail' + "_" + str(idx)] = subset_df
            subset_df.to_excel('Subset_Cluster_Sail' + "_" + str(idx) + '.xlsx', index = False)
            final_sail_df = pd.concat([final_sail_df, subset_df])
            
    if activity == 'PORT':
        final_port_df = pd.DataFrame()
        for idx in range(1, no_of_profiles+1):
            print("idx-p", idx)
            df = cluster_data[cluster_data['Cluster'] == idx].copy() 
            print("Data Availibility Days", round(df['activity_time'].sum()/24))
            
            subset_df = find_port_subset_data(df, expected_days)
            finaldict_port['Subset_Cluster_Port' + "_" + str(idx)] = subset_df  
            subset_df.to_excel('Subset_Cluster_Port' + "_" + str(idx) + '.xlsx', index = False)
            final_port_df = pd.concat([final_port_df, subset_df] )

In [24]:
get_random_sample_with_time_sum(sail_cluster, 'SAIL',  3,   120  )

idx-s 1
Data Availibility Days 189


AttributeError: 'DataFrame' object has no attribute 'append'

In [13]:
final_sail_df.to_excel("Subset_Sail_DF.xlsx", index = False)

In [ ]:
finaldict_sail.keys()

In [ ]:
get_random_sample_with_time_sum(port_cluster, 'PORT',  3,   60  )

In [ ]:
final_port_df.isnull().sum()

In [ ]:
final_port_df.to_excel("Subset_Port_DF.xlsx", index = False)

In [ ]:
finaldict_port.keys()

In [ ]:
final_port_df